In [0]:
import requests
import time

tickers_df = spark.sql("SELECT ticker FROM finance_intelligence_hub.bronze.companys")
full_tickers_list = [row['ticker'] for row in tickers_df.collect()]

# 🌟 اسم الجدول الجديد تماماً الخاص بالموقع والفيتشرز الكاملة
target_table_name = "finance_intelligence_hub.bronze.company_news_polygon"

# 2. فحص الجدول الجديد للاستكمال في حال حدوث Interrupt
try:
    saved_tickers_df = spark.sql(f"SELECT DISTINCT ticker FROM {target_table_name}")
    saved_tickers_list = set([row['ticker'] for row in saved_tickers_df.collect()])
    print(f"📦 وجدنا {len(saved_tickers_list)} شركة محفوظة بالفعل في الجدول الجديد.")
except Exception as e:
    saved_tickers_list = set()
    print("ℹ️ الجدول الجديد غير موجود أو فارغ، سيبدأ السحب الشامل وبناء الجدول من الصفر.")

# فلترة القائمة لاستبعاد الشركات المنجزة
tickers_list = [t for t in full_tickers_list if t not in saved_tickers_list]
total_companies = len(tickers_list)

# إعدادات الـ API والمفتاح الخاص بك
api_key = "9dsWtWB7r0h5ZXqzfxVcdo1VoGmHSF1o"
start_date = "2022-01-01"
end_date = "2026-07-02"

grand_total_saved = 0

print(f"=== 🔄 بدء بناء الجدول الجديد وضخ البيانات لـ {total_companies} شركة متبقية ===")

# 3. حلقة التكرار للمرور على الشركات
for index, ticker in enumerate(tickers_list, 1):
    print(f"\n🔄 [{index}/{total_companies}] جاري سحب كافة تفاصيل شركة: {ticker}...")
    
    url = "https://api.polygon.io/v2/reference/news"
    params = {
        "ticker": ticker,
        "published_utc.gte": start_date,
        "published_utc.lte": end_date,
        "order": "desc",
        "limit": 1000,       
        "apiKey": api_key
    }
    
    company_news_count = 0
    page = 1
    all_records = []
    
    while url:
        response = requests.get(url, params=params if "apiKey" in params else None)
        
        if response.status_code == 200:
            data = response.json()
            results = data.get('results', [])
            
            if not results:
                break
                
            for news in results:
                title = news.get('title', '')
                description = news.get('description', '')
                date = news.get('published_utc', '')
                full_text_to_analyze = f"{title}. {description}".strip()
                
                # استخراج بيانات الناشر والروابط والصور للموقع
                publisher_data = news.get('publisher', {})
                publisher_name = publisher_data.get('name', '')
                publisher_homepage = publisher_data.get('homepage_url', '')
                related_tickers = ",".join(news.get('tickers', []))
                
                if date.startswith(end_date):
                    print(f"   📌 خبر مكتشف اليوم لـ {ticker} ({date}): {title[:50]}...")
                
                all_records.append({
                    "ticker": ticker,
                    "published_date": date,
                    "title": title,
                    "description": description,
                    "text_for_finbert": full_text_to_analyze,
                    "author": news.get('author', ''),
                    "source_url": news.get('article_url', ''),
                    "image_url": news.get('image_url', ''),
                    "publisher_name": publisher_name,
                    "publisher_url": publisher_homepage,
                    "related_tickers": related_tickers
                })
            
            company_news_count += len(results)
            print(f"   📢 [الدفعة {page}] - تم سحب {len(results)} خبر لـ {ticker}.")
            
            page += 1
            
            next_url = data.get('next_url')
            if next_url:
                url = f"{next_url}&apiKey={api_key}"
                params = {}  
                time.sleep(12) # Rate Limit للحساب المجاني
            else:
                url = None
                
        elif response.status_code == 429:
            print("\n⚠️ [Rate Limit] تم الوصول لحد الدقيقة، جاري الانتظار 30 ثانية...")
            time.sleep(30)
        else:
            print(f"\n❌ خطأ أثناء جلب {ticker}: {response.status_code}")
            break
            
    # 🚀 الحفظ الذكي أولاً بأول لحماية الذاكرة والـ Session
    if all_records:
        spark_df = spark.createDataFrame(all_records)
        
        # إذا كان الجدول جديداً تماماً والـ saved_tickers_list فارغة، فإن أول شركة تقوم بإنشائه (overwrite) لتهيئة الأعمدة، ومن ثم نستخدم (append).
        # أما لو كنت تكمل بعد Interrupt (الجدول يحتوي على داتا سابقاً)، فسيستخدم الكود وضع (append) دائماً للحفاظ عليها.
        if len(saved_tickers_list) == 0 and index == 1:
            write_mode = "overwrite"
        else:
            write_mode = "append"
            
        spark_df.write \
            .format("delta") \
            .mode(write_mode) \
            .option("mergeSchema", "true") \
            .saveAsTable(target_table_name)
            
        grand_total_saved += len(all_records)
        all_records.clear() # تفريغ الذاكرة فوراً لراحة الـ Driver Node
        
    print(f"✅ انتهى حفظ وتفريغ بيانات {ticker} بالكامل في الجدول الجديد.")

print("\n" + "="*50)
print(f"🎉 تم إنهاء إنشاء وتغذية الجدول الجديد بنجاح!")
print(f"💾 اسم الجدول الجديد في الـ Catalog الخاص بك: {target_table_name}")
print(f"📊 إجمالي الأخبار المضافة في هذه الجلسة: {grand_total_saved} خبر بكافة الفيتشرز المتاحة للموقع.")
print("="*50)